In [ ]:
# Fixed Drive Mount Cell
# Use a clean mount point instead of /content/drive

from google.colab import drive
from pathlib import Path

MOUNT_POINT = Path("/content/gdrive_mount")

# Create clean mount folder
MOUNT_POINT.mkdir(parents=True, exist_ok=True)

# If already mounted, skip remount
DRIVE_ROOT = MOUNT_POINT / "MyDrive"

if DRIVE_ROOT.exists():
    print("Drive already mounted at:", MOUNT_POINT)
else:
    print("Mounting Google Drive at:", MOUNT_POINT)
    drive.mount(str(MOUNT_POINT), force_remount=True)

DRIVE_ROOT = MOUNT_POINT / "MyDrive"

print("DRIVE_ROOT:", DRIVE_ROOT)
print("DRIVE_ROOT exists:", DRIVE_ROOT.exists())

print("\nMyDrive contents:")
for item in DRIVE_ROOT.iterdir():
    print("-", item.name)

Mounting Google Drive at: /content/gdrive_mount
Mounted at /content/gdrive_mount
DRIVE_ROOT: /content/gdrive_mount/MyDrive
DRIVE_ROOT exists: True

MyDrive contents:
- PAB-01.pdf.gdrive
- Untitled folder
- Facebook marketing.gsheet
- DriveBuzz
- index.html
- Ashraful Islam.zip
- MFlixBD.COM | MLWBD.COM FANTASTIC BEASTS AND WHERE TO  FIND THEM (2016) DUAL AUDIO   HINDI DD -ENGLISH BLURAY480 [476MB] | MKV.mkv
- Ashraful video small.mp4
- MLWBD.com The Old Man And The Sea (1999) Fs.Dvdrip.-720p.mkv
- Classroom
- opensea
- Telegram
- Discord
- IMG_20231114_193619_708.jpg
- Question .gdoc
- Problem list -CSE222-Algorithm_2021-22.pdf.gdoc
- shakib final.docx.gdoc
- code (1).cpp.gdoc
- code.cpp.gdoc
- IMG_20231021_214733.jpg
- IMG_20231021_214731.jpg
- Screenshot_20250310-150322.png
- Airdrop Analyst.gdoc
- Wedding.gslides
- Science project.gslides
- ch:1, chapter:3
- Smart Lock System.gslides
- Jolly LLB 3 (2025) Hindi WEB-DL 720p HEVC.mkv
- MLWBD.com Uncharted.2022.WEB-DL.720p.Dual.mkv
- ML

In [ ]:
from pathlib import Path

PROJECT_DIR = Path("/content/gdrive_mount/MyDrive/Thesis/thesis_project")
DATA_DIR = PROJECT_DIR / "data"
MODEL_DIR = PROJECT_DIR / "02_after_pruning_quantization" / "models"
DATASET_ZIP_PATH = Path("/content/gdrive_mount/MyDrive/Thesis/Dataset/PlantVillage.zip")

paths = {
    "Train CSV": DATA_DIR / "train_files.csv",
    "Val CSV": DATA_DIR / "val_files.csv",
    "Dataset zip": DATASET_ZIP_PATH,
    "11% pruned model": MODEL_DIR / "mobilenetv2_safe_structured_pruned_11_percent_full_model.pth",
}

for name, path in paths.items():
    print(name, ":", path.exists(), "|", path)

Train CSV : True | /content/gdrive_mount/MyDrive/Thesis/thesis_project/data/train_files.csv
Val CSV : True | /content/gdrive_mount/MyDrive/Thesis/thesis_project/data/val_files.csv
Dataset zip : True | /content/gdrive_mount/MyDrive/Thesis/Dataset/PlantVillage.zip
11% pruned model : True | /content/gdrive_mount/MyDrive/Thesis/thesis_project/02_after_pruning_quantization/models/mobilenetv2_safe_structured_pruned_11_percent_full_model.pth


In [ ]:
# Cell 02: Import libraries, check GPU, and set project paths

from pathlib import Path
import os, copy, time, random, shutil, zipfile, warnings
import numpy as np
import pandas as pd
from PIL import Image
from datetime import datetime

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from tqdm.auto import tqdm

from torch.ao.quantization import get_default_qat_qconfig_mapping
from torch.ao.quantization.quantize_fx import prepare_qat_fx, convert_fx

warnings.filterwarnings("ignore")

SEED = 42
IMG_SIZE = 224
NUM_CLASSES = 38
BATCH_SIZE = 32

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

PROJECT_DIR = Path("/content/gdrive_mount/MyDrive/Thesis/thesis_project")
DATA_DIR = PROJECT_DIR / "data"
OPT_DIR = PROJECT_DIR / "02_after_pruning_quantization"
MODEL_DIR = OPT_DIR / "models"
RESULTS_DIR = OPT_DIR / "results"

DATASET_ZIP_PATH = Path("/content/gdrive_mount/MyDrive/Thesis/Dataset/PlantVillage.zip")
PRUNED_11_MODEL_PATH = MODEL_DIR / "mobilenetv2_safe_structured_pruned_11_percent_full_model.pth"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Train CSV exists:", (DATA_DIR / "train_files.csv").exists())
print("Val CSV exists:", (DATA_DIR / "val_files.csv").exists())
print("Dataset zip exists:", DATASET_ZIP_PATH.exists())
print("11% pruned model exists:", PRUNED_11_MODEL_PATH.exists())

print("\nTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("GPU is not available. Enable T4 GPU before running QAT.")

QAT_DEVICE = torch.device("cuda")
INT8_DEVICE = torch.device("cpu")

print("GPU:", torch.cuda.get_device_name(0))
print("QAT device:", QAT_DEVICE)
print("INT8 device:", INT8_DEVICE)

PREFERRED_BACKENDS = ["x86", "fbgemm", "qnnpack"]
SELECTED_QUANT_BACKEND = None

for backend in PREFERRED_BACKENDS:
    if backend in torch.backends.quantized.supported_engines:
        torch.backends.quantized.engine = backend
        SELECTED_QUANT_BACKEND = backend
        break

if SELECTED_QUANT_BACKEND is None:
    raise RuntimeError("No suitable quantization backend found.")

QAT_QCONFIG_MAPPING = get_default_qat_qconfig_mapping(SELECTED_QUANT_BACKEND)

print("Selected quantization backend:", SELECTED_QUANT_BACKEND)

Train CSV exists: True
Val CSV exists: True
Dataset zip exists: True
11% pruned model exists: True

Torch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
QAT device: cuda
INT8 device: cpu
Selected quantization backend: x86


In [ ]:
# Cell 03: Extract PlantVillage dataset and create dataloaders

LOCAL_ZIP_PATH = Path("/content/PlantVillage.zip")
EXTRACT_DIR = Path("/content/PlantVillage_extracted")

train_csv = DATA_DIR / "train_files.csv"
val_csv = DATA_DIR / "val_files.csv"

if not LOCAL_ZIP_PATH.exists() or LOCAL_ZIP_PATH.stat().st_size != DATASET_ZIP_PATH.stat().st_size:
    print("Copying PlantVillage.zip to local storage...")
    shutil.copy2(DATASET_ZIP_PATH, LOCAL_ZIP_PATH)
    print("Copy completed.")
else:
    print("Local PlantVillage.zip already exists.")

EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

if not any(EXTRACT_DIR.iterdir()):
    print("Extracting dataset...")
    with zipfile.ZipFile(LOCAL_ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)
    print("Extraction completed.")
else:
    print("Dataset already extracted.")

train_df_check = pd.read_csv(train_csv)
first_relative_path = train_df_check.iloc[0]["relative_path"]

candidate_roots = [
    EXTRACT_DIR,
    EXTRACT_DIR / "PlantVillage",
    EXTRACT_DIR / "PlantVillage" / "PlantVillage",
    EXTRACT_DIR / "PlantVillage" / "color",
    EXTRACT_DIR / "color",
]

DATASET_ROOT = None

for root in candidate_roots:
    test_path = root / first_relative_path
    print("Checking:", root, "| Exists:", test_path.exists())
    if test_path.exists():
        DATASET_ROOT = root
        break

if DATASET_ROOT is None:
    raise FileNotFoundError("DATASET_ROOT not detected.")

print("Final DATASET_ROOT:", DATASET_ROOT)


class PlantVillageCSVDataset(Dataset):
    def __init__(self, csv_path, dataset_root_dir, transform=None):
        self.csv_path = Path(csv_path)
        self.dataset_root_dir = Path(dataset_root_dir)
        self.transform = transform
        self.df = pd.read_csv(self.csv_path)

        self.path_col = "relative_path"
        self.label_col = "label"

        sample_path = self.dataset_root_dir / self.df.iloc[0][self.path_col]
        print("Sample image exists:", sample_path.exists())

        if not sample_path.exists():
            raise FileNotFoundError("Sample image not found. Check DATASET_ROOT.")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self.dataset_root_dir / row[self.path_col]
        label = int(row[self.label_col])

        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label


qat_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

qat_train_dataset = PlantVillageCSVDataset(
    csv_path=train_csv,
    dataset_root_dir=DATASET_ROOT,
    transform=qat_transform
)

qat_val_dataset = PlantVillageCSVDataset(
    csv_path=val_csv,
    dataset_root_dir=DATASET_ROOT,
    transform=qat_transform
)

qat_train_loader = DataLoader(
    qat_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

qat_val_loader_gpu = DataLoader(
    qat_val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

qat_full_val_loader_cpu = DataLoader(
    qat_val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

print("QAT training samples:", len(qat_train_dataset))
print("Validation samples:", len(qat_val_dataset))

Copying PlantVillage.zip to local storage...
Copy completed.
Extracting dataset...
Extraction completed.
Checking: /content/PlantVillage_extracted | Exists: False
Checking: /content/PlantVillage_extracted/PlantVillage | Exists: True
Final DATASET_ROOT: /content/PlantVillage_extracted/PlantVillage
Sample image exists: True
Sample image exists: True
QAT training samples: 40728
Validation samples: 8146


In [ ]:
# Cell 04: Define evaluation and QAT training functions

def evaluate_model_gpu(model, data_loader, device, desc="Evaluating on GPU"):
    model.eval()

    all_preds = []
    all_labels = []
    total_loss = 0.0
    criterion = nn.CrossEntropyLoss()

    with torch.no_grad():
        for images, labels in tqdm(data_loader, desc=desc):
            images = images.to(device, non_blocking=True).float()
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)
            loss = criterion(outputs.float(), labels)
            preds = torch.argmax(outputs, dim=1)

            total_loss += loss.item() * images.size(0)
            all_preds.extend(preds.detach().cpu().numpy())
            all_labels.extend(labels.detach().cpu().numpy())

    avg_loss = total_loss / len(data_loader.dataset)
    acc = accuracy_score(all_labels, all_preds)

    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels,
        all_preds,
        average="macro",
        zero_division=0
    )

    return {
        "loss": avg_loss,
        "accuracy": acc,
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": f1
    }


def evaluate_model_cpu(model, data_loader, desc="Evaluating on CPU"):
    model.eval()

    all_preds = []
    all_labels = []
    total_loss = 0.0
    criterion = nn.CrossEntropyLoss()

    with torch.no_grad():
        for images, labels in tqdm(data_loader, desc=desc):
            images = images.to(INT8_DEVICE).float()
            labels = labels.to(INT8_DEVICE)

            outputs = model(images)
            loss = criterion(outputs.float(), labels)
            preds = torch.argmax(outputs, dim=1)

            total_loss += loss.item() * images.size(0)
            all_preds.extend(preds.detach().cpu().numpy())
            all_labels.extend(labels.detach().cpu().numpy())

    avg_loss = total_loss / len(data_loader.dataset)
    acc = accuracy_score(all_labels, all_preds)

    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels,
        all_preds,
        average="macro",
        zero_division=0
    )

    return {
        "loss": avg_loss,
        "accuracy": acc,
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": f1
    }


QAT_EPOCHS = 3
QAT_LEARNING_RATE = 1e-6
QAT_WEIGHT_DECAY = 1e-5

BASELINE_VAL_ACCURACY = 0.9969310090842131
BASELINE_VAL_MACRO_F1 = 0.9965513617249061

PRUNED_FP32_VAL_METRICS = {
    "loss": 0.047960480862886366,
    "accuracy": 0.9862509206972747,
    "macro_precision": 0.9837854443512238,
    "macro_recall": 0.9818269836199642,
    "macro_f1": 0.9826975305474518
}


def train_one_epoch_qat(model, train_loader, optimizer, criterion, device, epoch, total_epochs):
    model.train()

    running_loss = 0.0
    all_preds = []
    all_labels = []

    progress_bar = tqdm(train_loader, desc=f"QAT training epoch {epoch}/{total_epochs}")

    for images, labels in progress_bar:
        images = images.to(device, non_blocking=True).float()
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        outputs = model(images)
        loss = criterion(outputs.float(), labels)

        loss.backward()
        optimizer.step()

        preds = torch.argmax(outputs, dim=1)

        running_loss += loss.item() * images.size(0)
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

        progress_bar.set_postfix({"loss": loss.item()})

    epoch_loss = running_loss / len(train_loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)

    return epoch_loss, epoch_acc


def run_qat_finetuning(model):
    criterion = nn.CrossEntropyLoss()

    optimizer = optim.AdamW(
        model.parameters(),
        lr=QAT_LEARNING_RATE,
        weight_decay=QAT_WEIGHT_DECAY
    )

    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=QAT_EPOCHS
    )

    best_val_acc = -1.0
    best_state_dict = None
    best_metrics = None
    history = []

    start_time = time.time()

    for epoch in range(1, QAT_EPOCHS + 1):
        train_loss, train_acc = train_one_epoch_qat(
            model=model,
            train_loader=qat_train_loader,
            optimizer=optimizer,
            criterion=criterion,
            device=QAT_DEVICE,
            epoch=epoch,
            total_epochs=QAT_EPOCHS
        )

        val_metrics = evaluate_model_gpu(
            model=model,
            data_loader=qat_val_loader_gpu,
            device=QAT_DEVICE,
            desc=f"QAT fake-quant GPU validation epoch {epoch}"
        )

        scheduler.step()

        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_accuracy": train_acc,
            "val_loss": val_metrics["loss"],
            "val_accuracy": val_metrics["accuracy"],
            "val_macro_precision": val_metrics["macro_precision"],
            "val_macro_recall": val_metrics["macro_recall"],
            "val_macro_f1": val_metrics["macro_f1"],
            "learning_rate": optimizer.param_groups[0]["lr"]
        }

        history.append(row)

        print("\nQAT epoch summary:")
        for k, v in row.items():
            print(k, ":", v)

        if val_metrics["accuracy"] > best_val_acc:
            best_val_acc = val_metrics["accuracy"]
            best_metrics = val_metrics
            best_state_dict = copy.deepcopy(model.state_dict())
            print("New best QAT fake-quant validation accuracy:", best_val_acc)

    total_time = time.time() - start_time

    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)

    return model, best_metrics, history, total_time

In [ ]:
# Cell 05: Run QAT training and immediately save fake-quant checkpoint

qat_base_checkpoint = torch.load(
    PRUNED_11_MODEL_PATH,
    map_location="cpu",
    weights_only=False
)

qat_base_model_fp32 = qat_base_checkpoint["model"]
qat_base_model_fp32 = qat_base_model_fp32.float().cpu()
qat_base_model_fp32.train()

print("Loaded 11% pruned model.")
print("Pruning ratio:", qat_base_checkpoint.get("pruning_ratio", "Not found"))

example_inputs_cpu = (
    torch.randn(1, 3, IMG_SIZE, IMG_SIZE),
)

prepared_qat_model = prepare_qat_fx(
    qat_base_model_fp32,
    QAT_QCONFIG_MAPPING,
    example_inputs_cpu
)

prepared_qat_model = prepared_qat_model.to(QAT_DEVICE)
prepared_qat_model.train()

print("QAT model prepared. Starting QAT training...")

qat_model_trained, qat_best_val_metrics, qat_history, qat_training_time = run_qat_finetuning(
    prepared_qat_model
)

qat_history_df = pd.DataFrame(qat_history)

QAT_HISTORY_PATH = RESULTS_DIR / "qat_11_percent_training_history_final_rerun.csv"
qat_history_df.to_csv(QAT_HISTORY_PATH, index=False)

QAT_TEMP_DIR = OPT_DIR / "quantization_experiments" / "qat_int8_11_percent" / "temporary_checkpoints"
QAT_TEMP_DIR.mkdir(parents=True, exist_ok=True)

QAT_FAKE_QUANT_PATH = QAT_TEMP_DIR / "qat_11_percent_fake_quant_best_final_rerun.pth"

qat_fake_quant_cpu = copy.deepcopy(qat_model_trained).cpu()

fake_quant_checkpoint = {
    "model": qat_fake_quant_cpu,
    "model_state_dict": qat_fake_quant_cpu.state_dict(),
    "model_name": "11% Pruned + QAT Fake-Quant MobileNetV2",
    "base_pruned_model_path": str(PRUNED_11_MODEL_PATH),
    "pruning_ratio": qat_base_checkpoint.get("pruning_ratio", None),
    "qat_epochs": QAT_EPOCHS,
    "qat_learning_rate": QAT_LEARNING_RATE,
    "qat_weight_decay": QAT_WEIGHT_DECAY,
    "qat_best_val_metrics": qat_best_val_metrics,
    "qat_history": qat_history,
    "quantization_backend": SELECTED_QUANT_BACKEND,
    "status": "fake_quant_qat_checkpoint_before_real_int8_conversion",
    "saved_at": datetime.now().isoformat()
}

torch.save(fake_quant_checkpoint, QAT_FAKE_QUANT_PATH)

print("\n✅ QAT training completed.")
print("QAT training time seconds:", qat_training_time)

print("\nBest QAT fake-quant validation metrics:")
for k, v in qat_best_val_metrics.items():
    print(k, ":", v)

print("\n✅ QAT history saved:")
print(QAT_HISTORY_PATH)

print("\n✅ QAT fake-quant checkpoint saved:")
print(QAT_FAKE_QUANT_PATH)

display(qat_history_df)

Loaded 11% pruned model.
Pruning ratio: 0.11
QAT model prepared. Starting QAT training...


QAT training epoch 1/3:   0%|          | 0/1273 [00:00<?, ?it/s]

QAT fake-quant GPU validation epoch 1:   0%|          | 0/255 [00:00<?, ?it/s]


QAT epoch summary:
epoch : 1
train_loss : 0.11405097831796508
train_accuracy : 0.9671479080730702
val_loss : 0.10514816390101224
val_accuracy : 0.9675914559292904
val_macro_precision : 0.9629781565480293
val_macro_recall : 0.9486197775994254
val_macro_f1 : 0.9547887670978318
learning_rate : 7.5e-07
New best QAT fake-quant validation accuracy: 0.9675914559292904


QAT training epoch 2/3:   0%|          | 0/1273 [00:00<?, ?it/s]

...

QAT training epoch 3/3:   0%|          | 0/1273 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7821468ecb80>Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7821468ecb80>
Traceback (most recent call last):

  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
Traceback (most recent call last):
      File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
self._shutdown_workers()    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers

  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
        if w.is_alive():if w.is_alive():

             ^ ^^^^^^^^^^^^^^^^^^^^^^^

  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
        assert self.

QAT fake-quant GPU validation epoch 3:   0%|          | 0/255 [00:00<?, ?it/s]


QAT epoch summary:
epoch : 3
train_loss : 0.09378848735889209
train_accuracy : 0.9716902376743273
val_loss : 0.09850615006156967
val_accuracy : 0.9694328504787626
val_macro_precision : 0.9662477128829686
val_macro_recall : 0.9554568373499156
val_macro_f1 : 0.9600551601295203
learning_rate : 0.0
New best QAT fake-quant validation accuracy: 0.9694328504787626


AttributeError: Can't get local object '_add_module_to_qconfig_obs_ctr.<locals>.get_factory_kwargs_based_on_module_device'

In [ ]:
# Cell 05B: Safely save QAT training results without saving the full model object
# This avoids PyTorch pickle errors by saving only state_dict, metrics, and history.

from datetime import datetime
from pathlib import Path
import torch
import pandas as pd

print("Checking required variables:")
print("qat_model_trained:", "qat_model_trained" in globals())
print("qat_best_val_metrics:", "qat_best_val_metrics" in globals())
print("qat_history:", "qat_history" in globals())

if "qat_model_trained" not in globals():
    raise RuntimeError("qat_model_trained is missing. Do not continue; QAT model is not in memory.")

QAT_TEMP_DIR = OPT_DIR / "quantization_experiments" / "qat_int8_11_percent" / "temporary_checkpoints"
QAT_TEMP_DIR.mkdir(parents=True, exist_ok=True)

QAT_STATE_DICT_PATH = QAT_TEMP_DIR / "qat_11_percent_fake_quant_best_state_dict_final_rerun.pth"
QAT_HISTORY_PATH = RESULTS_DIR / "qat_11_percent_training_history_final_rerun.csv"
QAT_SAFE_SUMMARY_PATH = RESULTS_DIR / "qat_11_percent_fake_quant_summary_final_rerun.csv"

# Save QAT state_dict only, not full Python model object
qat_state_checkpoint = {
    "model_state_dict": qat_model_trained.cpu().state_dict(),
    "model_name": "11% Pruned + QAT Fake-Quant MobileNetV2",
    "base_pruned_model_path": str(PRUNED_11_MODEL_PATH),
    "pruning_ratio": qat_base_checkpoint.get("pruning_ratio", None),
    "qat_epochs": QAT_EPOCHS,
    "qat_learning_rate": QAT_LEARNING_RATE,
    "qat_weight_decay": QAT_WEIGHT_DECAY,
    "qat_best_val_metrics": qat_best_val_metrics,
    "qat_history": qat_history,
    "quantization_backend": SELECTED_QUANT_BACKEND,
    "status": "fake_quant_qat_state_dict_saved_before_real_int8_conversion",
    "saved_at": datetime.now().isoformat(),
    "important_note": "This file saves state_dict only to avoid PyTorch FX/QAT full-object pickle issues."
}

torch.save(qat_state_checkpoint, QAT_STATE_DICT_PATH)

# Put model back to GPU if needed later
qat_model_trained = qat_model_trained.to(QAT_DEVICE)
qat_model_trained.eval()

qat_history_df = pd.DataFrame(qat_history)
qat_history_df.to_csv(QAT_HISTORY_PATH, index=False)

best_row = qat_history_df.sort_values(by="val_accuracy", ascending=False).iloc[0]

safe_summary = pd.DataFrame([{
    "model_name": "11% Pruned + QAT Fake-Quant MobileNetV2",
    "best_epoch": int(best_row["epoch"]),
    "best_val_accuracy": qat_best_val_metrics["accuracy"],
    "best_val_macro_f1": qat_best_val_metrics["macro_f1"],
    "best_val_loss": qat_best_val_metrics["loss"],
    "qat_epochs": QAT_EPOCHS,
    "qat_learning_rate": QAT_LEARNING_RATE,
    "state_dict_path": str(QAT_STATE_DICT_PATH),
    "history_path": str(QAT_HISTORY_PATH),
    "status": "QAT fake-quant training completed and state_dict saved"
}])

safe_summary.to_csv(QAT_SAFE_SUMMARY_PATH, index=False)

print("✅ QAT state_dict checkpoint saved safely:")
print(QAT_STATE_DICT_PATH)

print("\n✅ QAT history saved:")
print(QAT_HISTORY_PATH)

print("\n✅ QAT summary saved:")
print(QAT_SAFE_SUMMARY_PATH)

print("\nBest QAT fake-quant validation metrics:")
for k, v in qat_best_val_metrics.items():
    print(k, ":", v)

display(safe_summary)

Checking required variables:
qat_model_trained: True
qat_best_val_metrics: True
qat_history: True
✅ QAT state_dict checkpoint saved safely:
/content/gdrive_mount/MyDrive/Thesis/thesis_project/02_after_pruning_quantization/quantization_experiments/qat_int8_11_percent/temporary_checkpoints/qat_11_percent_fake_quant_best_state_dict_final_rerun.pth

✅ QAT history saved:
/content/gdrive_mount/MyDrive/Thesis/thesis_project/02_after_pruning_quantization/results/qat_11_percent_training_history_final_rerun.csv

✅ QAT summary saved:
/content/gdrive_mount/MyDrive/Thesis/thesis_project/02_after_pruning_quantization/results/qat_11_percent_fake_quant_summary_final_rerun.csv

Best QAT fake-quant validation metrics:
loss : 0.09850615006156967
accuracy : 0.9694328504787626
macro_precision : 0.9662477128829686
macro_recall : 0.9554568373499156
macro_f1 : 0.9600551601295203


,model_name,best_epoch,best_val_accuracy,best_val_macro_f1,best_val_loss,qat_epochs,qat_learning_rate,state_dict_path,history_path,status
0,11% Pruned + QAT Fake-Quant MobileNetV2,3,0.969433,0.960055,0.098506,3,0.000001,/content/gdrive_mount/MyDrive/Thesis/thesis_pr...,/content/gdrive_mount/MyDrive/Thesis/thesis_pr...,QAT fake-quant training completed and state_di...


In [ ]:
# Cell 06: Convert to real INT8, run full CPU validation, and save final TorchScript model
# This cell saves the final reliable model as TorchScript for local CPU evaluation.

import copy
import torch
import pandas as pd
from pathlib import Path
from datetime import datetime

# Make sure model is available
if "qat_model_trained" not in globals():
    raise RuntimeError("qat_model_trained is missing. You must rerun QAT training first.")

torch.backends.quantized.engine = SELECTED_QUANT_BACKEND

# Move trained QAT model to CPU and convert to real INT8
qat_model_for_convert = copy.deepcopy(qat_model_trained)
qat_model_for_convert = qat_model_for_convert.to(INT8_DEVICE)
qat_model_for_convert.eval()

qat_int8_model = convert_fx(qat_model_for_convert)
qat_int8_model.eval()

print("✅ QAT model converted to real INT8.")
print("Quantization backend:", SELECTED_QUANT_BACKEND)

# Full CPU validation
print("\nStarting FULL CPU validation...")
print("Total validation samples:", len(qat_val_dataset))
print("Total validation batches:", len(qat_full_val_loader_cpu))

qat_int8_full_val_metrics = evaluate_model_cpu(
    qat_int8_model,
    qat_full_val_loader_cpu,
    desc="Full validating QAT-INT8 model on CPU"
)

print("\nQAT-INT8 full validation metrics:")
for k, v in qat_int8_full_val_metrics.items():
    print(k, ":", v)

# Performance drop calculation
qat_int8_full_acc_drop_from_baseline_pp = (
    BASELINE_VAL_ACCURACY - qat_int8_full_val_metrics["accuracy"]
) * 100

qat_int8_full_f1_drop_from_baseline_pp = (
    BASELINE_VAL_MACRO_F1 - qat_int8_full_val_metrics["macro_f1"]
) * 100

qat_int8_full_acc_drop_from_pruned_fp32_pp = (
    PRUNED_FP32_VAL_METRICS["accuracy"] - qat_int8_full_val_metrics["accuracy"]
) * 100

qat_int8_full_f1_drop_from_pruned_fp32_pp = (
    PRUNED_FP32_VAL_METRICS["macro_f1"] - qat_int8_full_val_metrics["macro_f1"]
) * 100

print("\nAccuracy/F1 drop comparison:")
print("QAT-INT8 full accuracy drop from baseline:", qat_int8_full_acc_drop_from_baseline_pp, "percentage point")
print("QAT-INT8 full macro F1 drop from baseline:", qat_int8_full_f1_drop_from_baseline_pp, "percentage point")
print("QAT-INT8 full accuracy drop from 11% pruned FP32:", qat_int8_full_acc_drop_from_pruned_fp32_pp, "percentage point")
print("QAT-INT8 full macro F1 drop from 11% pruned FP32:", qat_int8_full_f1_drop_from_pruned_fp32_pp, "percentage point")

# Save final model with clear name
FINAL_MODEL_NAME = "Final_Optimized_MobileNetV2_11Percent_Pruned_QAT_INT8"

final_torchscript_path = MODEL_DIR / f"{FINAL_MODEL_NAME}_TorchScript.pt"
final_metadata_path = MODEL_DIR / f"{FINAL_MODEL_NAME}_metadata.pth"
final_summary_path = RESULTS_DIR / f"{FINAL_MODEL_NAME}_summary.csv"
final_report_path = RESULTS_DIR / f"{FINAL_MODEL_NAME}_report.md"

# Save TorchScript model
example_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(INT8_DEVICE)

traced_model = torch.jit.trace(qat_int8_model, example_input)
torch.jit.save(traced_model, str(final_torchscript_path))

def get_file_size_mb(path):
    path = Path(path)
    if path.exists():
        return path.stat().st_size / (1024 * 1024)
    return None

final_model_size_mb = get_file_size_mb(final_torchscript_path)

# Metadata without full model object
metadata = {
    "readable_model_name": "Final Optimized MobileNetV2 - 11% Pruned + QAT INT8",
    "file_model_name": FINAL_MODEL_NAME,
    "torchscript_model_path": str(final_torchscript_path),
    "optimization_pipeline": "11% Structured Pruning + QAT Fine-Tuning + INT8 Conversion",
    "quantization_type": "INT8_QAT_FX",
    "quantization_backend": SELECTED_QUANT_BACKEND,
    "pruning_ratio": qat_base_checkpoint.get("pruning_ratio", None),

    "baseline_val_accuracy": BASELINE_VAL_ACCURACY,
    "baseline_val_macro_f1": BASELINE_VAL_MACRO_F1,

    "pruned_fp32_val_metrics": PRUNED_FP32_VAL_METRICS,
    "qat_fake_quant_best_val_metrics": qat_best_val_metrics,
    "qat_int8_full_val_metrics": qat_int8_full_val_metrics,

    "qat_int8_full_accuracy_drop_from_baseline_percentage_point": qat_int8_full_acc_drop_from_baseline_pp,
    "qat_int8_full_f1_drop_from_baseline_percentage_point": qat_int8_full_f1_drop_from_baseline_pp,
    "qat_int8_full_accuracy_drop_from_pruned_fp32_percentage_point": qat_int8_full_acc_drop_from_pruned_fp32_pp,
    "qat_int8_full_f1_drop_from_pruned_fp32_percentage_point": qat_int8_full_f1_drop_from_pruned_fp32_pp,

    "final_torchscript_model_size_mb": final_model_size_mb,
    "saved_at": datetime.now().isoformat(),
    "important_note": (
        "Use the TorchScript .pt file for local CPU inference, latency, energy, and CO2 evaluation. "
        "Final thesis test-set evaluation must still be performed on the local Dell Latitude CPU device."
    )
}

torch.save(metadata, final_metadata_path)

summary_row = {
    "readable_model_name": "Final Optimized MobileNetV2 - 11% Pruned + QAT INT8",
    "file_model_name": FINAL_MODEL_NAME,
    "torchscript_model_path": str(final_torchscript_path),
    "metadata_path": str(final_metadata_path),
    "optimization_pipeline": "11% Structured Pruning + QAT Fine-Tuning + INT8 Conversion",
    "quantization_type": "INT8_QAT_FX",
    "quantization_backend": SELECTED_QUANT_BACKEND,
    "pruning_ratio": qat_base_checkpoint.get("pruning_ratio", None),

    "baseline_val_accuracy": BASELINE_VAL_ACCURACY,
    "baseline_val_macro_f1": BASELINE_VAL_MACRO_F1,

    "pruned_fp32_val_accuracy": PRUNED_FP32_VAL_METRICS["accuracy"],
    "pruned_fp32_val_macro_f1": PRUNED_FP32_VAL_METRICS["macro_f1"],

    "qat_fake_quant_best_val_accuracy": qat_best_val_metrics["accuracy"],
    "qat_fake_quant_best_val_macro_f1": qat_best_val_metrics["macro_f1"],

    "qat_int8_full_val_accuracy": qat_int8_full_val_metrics["accuracy"],
    "qat_int8_full_val_macro_f1": qat_int8_full_val_metrics["macro_f1"],

    "qat_int8_full_accuracy_drop_from_baseline_percentage_point": qat_int8_full_acc_drop_from_baseline_pp,
    "qat_int8_full_f1_drop_from_baseline_percentage_point": qat_int8_full_f1_drop_from_baseline_pp,

    "qat_int8_full_accuracy_drop_from_pruned_fp32_percentage_point": qat_int8_full_acc_drop_from_pruned_fp32_pp,
    "qat_int8_full_f1_drop_from_pruned_fp32_percentage_point": qat_int8_full_f1_drop_from_pruned_fp32_pp,

    "final_torchscript_model_size_mb": final_model_size_mb,
    "next_step": "Run final local CPU test-set evaluation for accuracy, latency, energy, CO2, and model size comparison."
}

summary_df = pd.DataFrame([summary_row])
summary_df.to_csv(final_summary_path, index=False)

report_text = f"""# Final Optimized Model Report

## Final Model
**Final Optimized MobileNetV2 - 11% Pruned + QAT INT8**

## Saved TorchScript Model
`{final_torchscript_path}`

## Optimization Pipeline
Baseline MobileNetV2 → 11% Structured Pruning → QAT Fine-Tuning → INT8 Conversion

## Validation Results

| Model | Validation Accuracy | Macro F1-score |
|---|---:|---:|
| Baseline MobileNetV2 | {BASELINE_VAL_ACCURACY:.6f} | {BASELINE_VAL_MACRO_F1:.6f} |
| 11% Pruned FP32 MobileNetV2 | {PRUNED_FP32_VAL_METRICS["accuracy"]:.6f} | {PRUNED_FP32_VAL_METRICS["macro_f1"]:.6f} |
| QAT Fake-Quant Model | {qat_best_val_metrics["accuracy"]:.6f} | {qat_best_val_metrics["macro_f1"]:.6f} |
| Final QAT-INT8 TorchScript Model | {qat_int8_full_val_metrics["accuracy"]:.6f} | {qat_int8_full_val_metrics["macro_f1"]:.6f} |

## Performance Drop from Baseline
- Accuracy drop: {qat_int8_full_acc_drop_from_baseline_pp:.4f} percentage points
- Macro F1 drop: {qat_int8_full_f1_drop_from_baseline_pp:.4f} percentage points

## Performance Drop from 11% Pruned FP32 Model
- Accuracy drop: {qat_int8_full_acc_drop_from_pruned_fp32_pp:.4f} percentage points
- Macro F1 drop: {qat_int8_full_f1_drop_from_pruned_fp32_pp:.4f} percentage points

## Final Model Size
- TorchScript model size: {final_model_size_mb:.4f} MB

## Next Step
Run final local CPU test-set evaluation using the same test images, preprocessing, batch size, and energy measurement procedure as the baseline.
"""

with open(final_report_path, "w", encoding="utf-8") as f:
    f.write(report_text)

print("\n✅ Final optimized model saved successfully.")
print("TorchScript model:")
print(final_torchscript_path)

print("\nMetadata:")
print(final_metadata_path)

print("\nSummary CSV:")
print(final_summary_path)

print("\nMarkdown report:")
print(final_report_path)

print("\nFinal TorchScript model size MB:", final_model_size_mb)

display(summary_df)

✅ QAT model converted to real INT8.
Quantization backend: x86

Starting FULL CPU validation...
Total validation samples: 8146
Total validation batches: 255


Full validating QAT-INT8 model on CPU:   0%|          | 0/255 [00:00<?, ?it/s]


QAT-INT8 full validation metrics:
loss : 0.0930905478674687
accuracy : 0.9711514853916032
macro_precision : 0.967156169923747
macro_recall : 0.9557936969680538
macro_f1 : 0.9607977274363783

Accuracy/F1 drop comparison:
QAT-INT8 full accuracy drop from baseline: 2.577952369260983 percentage point
QAT-INT8 full macro F1 drop from baseline: 3.5753634288527847 percentage point
QAT-INT8 full accuracy drop from 11% pruned FP32: 1.5099435305671483 percentage point
QAT-INT8 full macro F1 drop from 11% pruned FP32: 2.1899803111073535 percentage point

✅ Final optimized model saved successfully.
TorchScript model:
/content/gdrive_mount/MyDrive/Thesis/thesis_project/02_after_pruning_quantization/models/Final_Optimized_MobileNetV2_11Percent_Pruned_QAT_INT8_TorchScript.pt

Metadata:
/content/gdrive_mount/MyDrive/Thesis/thesis_project/02_after_pruning_quantization/models/Final_Optimized_MobileNetV2_11Percent_Pruned_QAT_INT8_metadata.pth

Summary CSV:
/content/gdrive_mount/MyDrive/Thesis/thesis_pro

,readable_model_name,file_model_name,torchscript_model_path,metadata_path,optimization_pipeline,quantization_type,quantization_backend,pruning_ratio,baseline_val_accuracy,baseline_val_macro_f1,...,qat_fake_quant_best_val_accuracy,qat_fake_quant_best_val_macro_f1,qat_int8_full_val_accuracy,qat_int8_full_val_macro_f1,qat_int8_full_accuracy_drop_from_baseline_percentage_point,qat_int8_full_f1_drop_from_baseline_percentage_point,qat_int8_full_accuracy_drop_from_pruned_fp32_percentage_point,qat_int8_full_f1_drop_from_pruned_fp32_percentage_point,final_torchscript_model_size_mb,next_step
0,Final Optimized MobileNetV2 - 11% Pruned + QAT...,Final_Optimized_MobileNetV2_11Percent_Pruned_Q...,/content/gdrive_mount/MyDrive/Thesis/thesis_pr...,/content/gdrive_mount/MyDrive/Thesis/thesis_pr...,11% Structured Pruning + QAT Fine-Tuning + INT...,INT8_QAT_FX,x86,0.11,0.996931,0.996551,...,0.969433,0.960055,0.971151,0.960798,2.577952,3.575363,1.509944,2.18998,2.413846,Run final local CPU test-set evaluation for ac...
